<a href="https://colab.research.google.com/github/e3la/i2dc/blob/main/Reels_Metadata_Review_and_Triage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Before you begin go up to Runtime and change the runtime type to T4 GPU - this will allow you to run the AI model whisper inside this colab to get subtitle files (SRT files) for enhanced accessiblity of spoken words in your videos.

In [ ]:
# @title <h1>Step 1: Provide Package & Load Progress</h1>
# @markdown Run this cell to set up the environment, load your package, and optionally resume a previous session.

# --- Installations ---
print("--- Setting up environment ---")
print("Installing required Python packages and ffmpeg...")
!pip install -q Pillow openpyxl pandas
!apt-get -qq install ffmpeg > /dev/null
print("✅ Environment is ready.")

import os
import shutil
import zipfile
import pandas as pd
from google.colab import files, drive
import glob
from IPython import get_ipython
from PIL import Image, ImageDraw
import subprocess # Needed for running ffmpeg

# --- Define constants ---
REVIEW_DIR = "/content/review_data"
METADATA_FILENAME_PATTERN = "*metadata.xlsx"
GDRIVE_I2DC_PATH = "/content/drive/MyDrive/i2dc"
SESSION_FILE_PATH = os.path.join(GDRIVE_I2DC_PATH, "review_session.pkl")
VIDEO_PLACEHOLDER_PATH = "/content/video_placeholder.png" # Path for our generated placeholder
METADATA_FILE_PATH = None
zip_filepath = None
df = None
using_gdrive = False

# --- Helper functions ---
def create_video_placeholder(path, width=400, height=225):
    """Creates a generic video placeholder image."""
    img = Image.new('RGB', (width, height), color = '#E0E0E0') # Light grey background
    d = ImageDraw.Draw(img)
    triangle_size = 40
    center_x, center_y = width // 2, height // 2
    p1 = (center_x - triangle_size // 2, center_y - triangle_size // 2)
    p2 = (center_x - triangle_size // 2, center_y + triangle_size // 2)
    p3 = (center_x + triangle_size // 2, center_y)
    d.polygon([p1, p2, p3], fill = '#FFFFFF', outline = '#BDBDBD')
    img.save(path)

def generate_thumbnail(video_path, output_path):
    """Generates a thumbnail for a video file using ffmpeg."""
    try:
        command = ['ffmpeg', '-ss', '00:00:01.00', '-i', video_path, '-vframes', '1', '-q:v', '2', '-y', output_path]
        subprocess.run(command, check=True, capture_output=True, text=True)
        return True
    except (subprocess.CalledProcessError, FileNotFoundError):
        return False

# --- Clean up previous local session & create placeholder ---
if os.path.exists(REVIEW_DIR):
    shutil.rmtree(REVIEW_DIR)
os.makedirs(REVIEW_DIR, exist_ok=True)
create_video_placeholder(VIDEO_PLACEHOLDER_PATH)


# --- Ask user for file source ---
while True:
    print("-" * 50)
    method = input(
        "How do you want to provide the package ZIP file?\n"
        "1. Upload directly to Colab.\n"
        "2. Use Google Drive (enables Save/Load Progress).\n"
        "Enter choice (1 or 2): "
    ).strip()
    if method in ['1', '2']:
        break
    else:
        print("Invalid choice. Please enter 1 or 2.")
print("-" * 50)

# --- Handle file source (omitted for brevity, this part is unchanged) ---
if method == '1':
    print("Selected: Upload directly to Colab.")
    try:
        uploaded = files.upload()
        if uploaded:
            uploaded_filename = list(uploaded.keys())[0]
            zip_filepath = os.path.join("/content", uploaded_filename)
            print(f"\nSuccessfully uploaded: '{uploaded_filename}'")
    except Exception as e:
        print(f"\nAn error occurred during upload: {e}")
elif method == '2':
    using_gdrive = True
    print("Selected: Use Google Drive.")
    try:
        drive.mount('/content/drive', force_remount=True)
        print("Google Drive mounted successfully.")
        if not os.path.isdir(GDRIVE_I2DC_PATH):
            print(f"\n❌ ERROR: The folder '{GDRIVE_I2DC_PATH}' does not exist.")
            print("Please create a folder named 'i2dc' in 'My Drive'.")
        else:
            zip_files_found = [f for f in os.listdir(GDRIVE_I2DC_PATH) if f.lower().endswith('.zip')]
            if len(zip_files_found) == 0:
                print(f"No .zip files found in '{GDRIVE_I2DC_PATH}'.")
            elif len(zip_files_found) == 1:
                zip_filename = zip_files_found[0]
                zip_filepath = os.path.join(GDRIVE_I2DC_PATH, zip_filename)
                print(f"Found unique ZIP file: '{zip_filename}'")
            else: # Handle multiple zips
                print("\nMultiple .zip files found. Please choose one:")
                for i, filename in enumerate(zip_files_found): print(f"  {i+1}. {filename}")
                choice = int(input(f"Enter the number (1-{len(zip_files_found)}): ")) - 1
                chosen_filename = zip_files_found[choice]
                zip_filepath = os.path.join(GDRIVE_I2DC_PATH, chosen_filename)
                print(f"You selected: '{chosen_filename}'")
    except Exception as e:
        print(f"\nAn error occurred with Google Drive: {e}")

# --- Unzip and Load Data ---
if zip_filepath and os.path.exists(zip_filepath):
    print(f"\n🔄 Unzipping {os.path.basename(zip_filepath)}...")
    with zipfile.ZipFile(zip_filepath, 'r') as z: z.extractall(REVIEW_DIR)
    print("✅ Unzip complete.")

    resumed_session = False
    if using_gdrive and os.path.exists(SESSION_FILE_PATH):
        print("\n--- Found a saved review session! ---")
        resume_choice = input("Do you want to load your previous progress? (y/n): ").lower().strip()
        if resume_choice == 'y':
            try:
                df = pd.read_pickle(SESSION_FILE_PATH)
                resumed_session = True
                print(f"✅ Successfully resumed session with {len(df)} records.")
            except Exception as e:
                print(f"❌ Failed to load session file: {e}. Loading from original Excel instead.")

    if not resumed_session:
        search_path = os.path.join(REVIEW_DIR, METADATA_FILENAME_PATTERN)
        metadata_files_found = glob.glob(search_path)
        if not metadata_files_found:
            print(f"\n❌ ERROR: Could not find a metadata file in the ZIP.")
        else:
            METADATA_FILE_PATH = metadata_files_found[0]
            print(f"\n📊 Loading new session from: {os.path.basename(METADATA_FILE_PATH)}")
            df = pd.read_excel(METADATA_FILE_PATH)
            for col in ['additional_files', 'cover_image_url', 'title', 'keywords', 'abstract']:
                if col not in df.columns: df[col] = ''
                else: df[col] = df[col].astype(str)
            df.fillna('', inplace=True)
            df['_is_removed'] = False
            df['_needs_alt_text'] = False
            df['_needs_srt'] = False
            print("✅ New session started.")

    # --- MODIFICATION: Pre-process all videos to generate thumbnails ---
    if df is not None:
        print("\n--- Pre-processing: Generating Video Thumbnails ---")
        videos_to_process = []
        for index, row in df.iterrows():
            media_filename = str(row.get('fulltext_url', ''))
            if any(media_filename.lower().endswith(ext) for ext in ['.mp4', '.mov', '.webm']):
                # Only process if it doesn't already have a valid cover image
                cover_image = str(row.get('cover_image_url', ''))
                if not cover_image or not os.path.exists(os.path.join(REVIEW_DIR, cover_image)):
                    videos_to_process.append((index, media_filename))

        if not videos_to_process:
            print("✅ No new video thumbnails needed.")
        else:
            total_videos = len(videos_to_process)
            for i, (index, media_filename) in enumerate(videos_to_process):
                print(f"  ({i+1}/{total_videos}) Generating thumbnail for '{media_filename}'...")
                video_filepath = os.path.join(REVIEW_DIR, media_filename)
                thumb_filename = f"{os.path.splitext(os.path.basename(media_filename))[0]}_thumb.jpg"
                thumb_path = os.path.join(REVIEW_DIR, thumb_filename)

                if os.path.exists(video_filepath) and generate_thumbnail(video_filepath, thumb_path):
                    df.loc[index, 'cover_image_url'] = thumb_filename
            print("✅ Thumbnail generation complete.")
        # --- END MODIFICATION ---

        print("\n---> You are now ready to proceed to Step 2: Interactive Review.")
    else:
        print("\n❌ Failed to load any data. Cannot proceed.")
else:
    print("\nNo valid ZIP file was provided. Cannot proceed.")

--- Setting up environment ---
Installing required Python packages and ffmpeg...
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_

In [ ]:
# @title <h1>Step 2: Interactive Triage & Review</h1>
# @markdown Run this cell to review metadata, edit SRT files, and flag items for AI processing.
import ipywidgets as widgets
from IPython.display import display, clear_output, Image, Video
import pandas as pd
import time

if 'df' not in locals() or df is None:
    print("❌ Metadata DataFrame is not loaded. Please successfully run Step 1 first.")
else:
    # --- Global state ---
    current_index = 0

    # --- UI Widget Definitions ---
    field_layout = widgets.Layout(width='95%')
    textarea_layout = widgets.Layout(width='95%', height='120px')
    title_widget = widgets.Text(description='Title:', layout=field_layout, style={'description_width': 'initial'})
    abstract_widget = widgets.Textarea(description='Post Text (in quotes):', layout=textarea_layout, style={'description_width': 'initial'})
    description_widget = widgets.Textarea(description='Description:', layout=textarea_layout, placeholder='Optional: Add a description here. It will be appended to the Post Text.', style={'description_width': 'initial'})
    keywords_widget = widgets.Text(description='Keywords:', layout=field_layout, placeholder='comma, separated, values', style={'description_width': 'initial'})
    needs_srt_checkbox = widgets.Checkbox(value=False, description='Flag for AI SRT Generation?', indent=False)
    needs_alt_text_checkbox = widgets.Checkbox(value=False, description='Flag for AI Alt-Text Gen?', indent=False)
    ai_flags_box = widgets.VBox([needs_alt_text_checkbox, needs_srt_checkbox], layout={'border': '1px solid #cccccc', 'padding': '10px'})
    srt_textarea_layout = widgets.Layout(width='95%', height='150px')
    srt_content_display = widgets.Textarea(description='SRT Content (view/edit):', layout=srt_textarea_layout, style={'description_width': 'initial'})
    add_srt_button = widgets.Button(description='Add/Replace SRT', button_style='info', icon='upload')
    remove_srt_button = widgets.Button(description='Remove SRT File', button_style='warning', icon='unlink')
    srt_button_box = widgets.HBox([remove_srt_button, add_srt_button])
    srt_review_box = widgets.VBox([srt_content_display, srt_button_box], layout={'border': '1px solid #cccccc', 'padding': '10px'})
    save_progress_button = widgets.Button(description="Save Progress", icon="save", button_style='success')
    session_status_label = widgets.Label(value="")
    prev_button, next_button = widgets.Button(description='< Previous'), widgets.Button(description='Next >')
    remove_button, restore_button = widgets.Button(description='Remove this Item', button_style='danger', icon='trash'), widgets.Button(description='Restore this Item', button_style='success', icon='undo', layout={'display': 'none'})
    progress_label, goto_input, goto_button = widgets.Label(), widgets.IntText(description='Go to:', value=1), widgets.Button(description='Go')
    media_output = widgets.Output()
    load_full_video_button = widgets.Button(description='Load Full Video', icon='play', button_style='primary', layout={'display':'none'})
    media_area = widgets.VBox([media_output, load_full_video_button], layout={'align_items': 'center', 'width': '420px'})

    # --- Helper & Logic Functions ---
    def find_next_valid_index(start_index):
        valid_indices = df[~df['_is_removed']].index
        return valid_indices[valid_indices > start_index][0] if any(valid_indices > start_index) else valid_indices[0] if any(valid_indices) else None

    def find_prev_valid_index(start_index):
        valid_indices = df[~df['_is_removed']].index
        return valid_indices[valid_indices < start_index][-1] if any(valid_indices < start_index) else valid_indices[-1] if any(valid_indices) else None

    # --- MODIFICATION: Updated save logic with HTML formatting ---
    def save_current_record():
        if current_index is not None and not df.loc[current_index, '_is_removed']:
            df.loc[current_index, 'title'] = title_widget.value
            df.loc[current_index, 'keywords'] = keywords_widget.value
            df.loc[current_index, '_needs_srt'] = needs_srt_checkbox.value
            df.loc[current_index, '_needs_alt_text'] = needs_alt_text_checkbox.value

            post_text = abstract_widget.value.strip()
            post_description = description_widget.value.strip()

            # Build the final abstract with HTML formatting
            final_abstract = f'📝 Post Text: "{post_text}"'

            if post_description:
                final_abstract += f'<p>Post Description: {post_description}'

            df.loc[current_index, 'abstract'] = final_abstract

    # --- MODIFICATION: Updated display logic to parse HTML abstract ---
    def display_record(index):
        global current_index
        current_index = index
        if current_index is None:
            with media_output: clear_output(wait=True); print("No items to review.")
            progress_label.value = "No items to review"
            return

        record = df.iloc[current_index]
        total_rem = (~df['_is_removed']).sum()
        progress_label.value = f'Item: {current_index + 1} of {len(df)} (Remaining: {total_rem})'
        goto_input.value = current_index + 1
        session_status_label.value = ""
        is_removed = bool(record['_is_removed'])
        all_editors = [title_widget, abstract_widget, description_widget, keywords_widget, needs_srt_checkbox, needs_alt_text_checkbox, srt_content_display, add_srt_button, remove_srt_button]
        for w in all_editors: w.disabled = is_removed
        remove_button.layout.display, restore_button.layout.display = ('none', 'flex') if is_removed else ('flex', 'none')

        # Set standard fields
        title_widget.value = record.get('title', '')
        keywords_widget.value = record.get('keywords', '')

        # --- New Parsing Logic for Abstract and Description ---
        abstract_widget.value = ''
        description_widget.value = ''
        raw_abstract = str(record.get('abstract', ''))

        # Check for new HTML format
        if raw_abstract.startswith('📝 Post Text: "'):
            parts = raw_abstract.split('<p>')
            # Parse Post Text from the first part
            post_text_part = parts[0]
            start_idx = post_text_part.find('"') + 1
            end_idx = post_text_part.rfind('"')
            if start_idx > 0 and end_idx > start_idx:
                abstract_widget.value = post_text_part[start_idx:end_idx]

            # Parse Post Description if it exists
            if len(parts) > 1:
                post_desc_part = parts[1]
                desc_start_str = 'Post Description: '
                start_idx = post_desc_part.find(desc_start_str)
                if start_idx != -1:
                    description_widget.value = post_desc_part[start_idx + len(desc_start_str):].strip()
        # Handle legacy/plain text format
        else:
            abstract_widget.value = raw_abstract
        # --- End New Parsing Logic ---

        media_filename = record.get('fulltext_url', '')
        is_video = any(media_filename.lower().endswith(ext) for ext in ['.mp4', '.mov', '.webm'])
        is_image = any(media_filename.lower().endswith(ext) for ext in ['.jpg', '.jpeg', '.png', '.gif', '.webp'])
        needs_alt_text_checkbox.layout.display = 'flex' if is_image else 'none'
        srt_review_box.layout.display = 'flex' if is_video else 'none'
        needs_alt_text_checkbox.value = bool(record.get('_needs_alt_text', False))
        needs_srt_checkbox.value = bool(record.get('_needs_srt', False))

        if is_video:
            srt_content_display.disabled = needs_srt_checkbox.value or is_removed
            add_srt_button.disabled = needs_srt_checkbox.value or is_removed
            srt_filenames = [f.strip() for f in str(record.get('additional_files', '')).split('|') if f.strip().lower().endswith('.srt')]
            if srt_filenames:
                srt_path = os.path.join(REVIEW_DIR, srt_filenames[0])
                if os.path.exists(srt_path):
                    with open(srt_path, 'r', encoding='utf-8', errors='ignore') as f: srt_content_display.value = f.read()
                    remove_srt_button.layout.display = 'flex'
                else:
                    srt_content_display.value = f"--- SRT FILE '{srt_filenames[0]}' NOT FOUND ---"; remove_srt_button.layout.display = 'none'
            else:
                srt_content_display.value = "--- No SRT file associated with this record. ---"; remove_srt_button.layout.display = 'none'

        with media_output:
            clear_output(wait=True)
            if not media_filename: print("No media file specified for this record.")
            else:
                if is_image:
                    load_full_video_button.layout.display = 'none'
                    media_filepath = os.path.join(REVIEW_DIR, media_filename)
                    if os.path.exists(media_filepath): display(Image(filename=media_filepath, width=400))
                    else: print(f"❌ Image not found: {media_filepath}")
                elif is_video:
                    load_full_video_button.layout.display = 'flex'
                    cover_image_path = os.path.join(REVIEW_DIR, record.get('cover_image_url', ''))
                    if record.get('cover_image_url') and os.path.exists(cover_image_path):
                        display(Image(filename=cover_image_path, width=400))
                    else:
                        display(Image(filename=VIDEO_PLACEHOLDER_PATH))
                else:
                    load_full_video_button.layout.display = 'none'; print(f"Unsupported file type: {media_filename}")

    def on_load_full_video_clicked(b):
        with media_output:
            clear_output(wait=True); record = df.iloc[current_index]
            video_path = os.path.join(REVIEW_DIR, record.get('fulltext_url', ''))
            if os.path.exists(video_path): display(Video(video_path, width=400, embed=True)); load_full_video_button.layout.display = 'none'
            else: print(f"❌ Video file not found: {video_path}")
    def on_nav_clicked(b): save_current_record(); display_record(find_next_valid_index(current_index) if b.description == 'Next >' else find_prev_valid_index(current_index))
    def on_goto_clicked(b): save_current_record(); display_record(max(0, min(len(df) - 1, goto_input.value - 1)))
    def on_remove_restore_clicked(b):
        is_restoring = (b.description == 'Restore this Item'); df.loc[current_index, '_is_removed'] = not is_restoring
        display_record(current_index)
        if not is_restoring: on_nav_clicked(next_button)
    def on_save_progress_clicked(b):
        b.disabled = True; session_status_label.value = "Saving..."
        save_current_record(); df.to_pickle(SESSION_FILE_PATH)
        session_status_label.value = f"✅ Progress saved to Drive at {time.strftime('%H:%M:%S')}."; b.disabled = False
    def on_add_srt_clicked(b):
        try:
            with media_output: clear_output(wait=True);
            uploaded = files.upload()
            if not uploaded: display_record(current_index); return
            uploaded_filename = list(uploaded.keys())[0]
            if not uploaded_filename.lower().endswith('.srt'):
                os.remove(os.path.join("/content", uploaded_filename)); session_status_label.value = f"❌ Not a .srt file."
                display_record(current_index); return
            shutil.move(os.path.join("/content", uploaded_filename), os.path.join(REVIEW_DIR, uploaded_filename))
            all_files = [f.strip() for f in str(df.loc[current_index, 'additional_files']).split('|') if f.strip() and not f.strip().lower().endswith('.srt')]
            all_files.append(uploaded_filename); df.loc[current_index, 'additional_files'] = '|'.join(all_files)
            df.loc[current_index, '_needs_srt'] = False; session_status_label.value = f"✅ Added '{uploaded_filename}'."; display_record(current_index)
        except Exception as e: session_status_label.value = f"An error occurred: {e}"; display_record(current_index)
    def on_remove_srt_clicked(b):
        all_files = [f.strip() for f in str(df.loc[current_index, 'additional_files']).split('|') if f.strip() and not f.strip().lower().endswith('.srt')]
        df.loc[current_index, 'additional_files'] = '|'.join(all_files); session_status_label.value = f"✅ SRT removed from record."; display_record(current_index)

    # --- Link Handlers ---
    next_button.on_click(on_nav_clicked); prev_button.on_click(on_nav_clicked); goto_button.on_click(on_goto_clicked)
    remove_button.on_click(on_remove_restore_clicked); restore_button.on_click(on_remove_restore_clicked)
    save_progress_button.on_click(on_save_progress_clicked); add_srt_button.on_click(on_add_srt_clicked)
    remove_srt_button.on_click(on_remove_srt_clicked); load_full_video_button.on_click(on_load_full_video_clicked)

    # --- Assemble UI with the new description box ---
    nav_controls = widgets.HBox([prev_button, progress_label, next_button], layout=widgets.Layout(justify_content='space-around'))
    goto_controls = widgets.HBox([goto_input, goto_button])
    top_controls = widgets.HBox([nav_controls, goto_controls, remove_button, restore_button], layout=widgets.Layout(justify_content='space-between', align_items='center'))
    metadata_editors = widgets.VBox([title_widget, abstract_widget, description_widget, keywords_widget])
    right_panel = widgets.VBox([widgets.HBox([save_progress_button, session_status_label]), metadata_editors, ai_flags_box, srt_review_box])
    main_review_area = widgets.HBox([media_area, right_panel], layout=widgets.Layout(align_items='flex-start'))
    reviewer_ui = widgets.VBox([top_controls, main_review_area])

    # --- Initial display ---
    display(reviewer_ui)
    display_record(find_next_valid_index(-1))

In [ ]:
# @title <h1>Step 3: Finalize and Split Packages</h1>
# @markdown Run this cell to save your work into two packages:
# @markdown 1. A package of items **ready for DC review** (downloaded).
# @markdown 2. A package of items needing AI review (saved to Google Drive `i2dc` folder).

import pandas as pd
from google.colab import files
import os
import shutil
import zipfile

# This function is already correct and does not need modification.
# It checks the 'cover_image_url' column by default.
def create_package(df_to_package, staging_dir, zip_base_name, keep_ai_flags=False):
    """Helper function to create a zip package from a dataframe."""
    if os.path.exists(staging_dir): shutil.rmtree(staging_dir)
    os.makedirs(staging_dir)

    for index, row in df_to_package.iterrows():
        # Copy all files listed in key columns, including the new 'cover_image_url'.
        for col in ['fulltext_url', 'additional_files', 'cover_image_url']:
             if pd.notna(row[col]) and row[col]:
                for filename in str(row[col]).split('|'):
                    filename = filename.strip()
                    if not filename: continue
                    source_path = os.path.join(REVIEW_DIR, filename)
                    if os.path.exists(source_path):
                        shutil.copy2(source_path, os.path.join(staging_dir, os.path.basename(filename)))
                    else:
                        print(f"  ⚠️ WARNING: File '{filename}' not found. It will not be included in the package.")

    columns_to_drop = ['_is_removed']
    if not keep_ai_flags:
        columns_to_drop.extend(['_needs_srt', '_needs_alt_text'])

    # The 'cover_image_url' column is preserved and saved to the Excel file.
    final_df = df_to_package.drop(columns=columns_to_drop, errors='ignore')
    final_df.to_excel(os.path.join(staging_dir, "metadata.xlsx"), index=False, engine='openpyxl')

    zip_path = shutil.make_archive(zip_base_name, 'zip', staging_dir)
    return zip_path

if 'df' not in locals() or df is None:
    print("❌ DataFrame not available. Cannot save. Please run Steps 1 and 2 first.")
else:
    # This call to save_current_record() is crucial to save the last item viewed
    if 'current_index' in locals() and current_index is not None:
        save_current_record()
        print("Final check: All changes from the current view are saved to memory.")
    else:
        print("Starting finalization process.")


    # Filter out removed items for all subsequent operations
    df_final = df[~df['_is_removed']].copy()

    # --- Filter DataFrames for splitting ---
    df_ai_needed = df_final[(df_final['_needs_srt'] == True) | (df_final['_needs_alt_text'] == True)]
    df_ready_for_review = df_final[(df_final['_needs_srt'] == False) & (df_final['_needs_alt_text'] == False)]

    print("\n" + "-" * 50)
    print("Review Summary:")
    print(f"  - {len(df_ready_for_review)} items are ready for DC review.")
    print(f"  - {len(df_ai_needed)} items flagged for AI processing.")
    print(f"  - {df['_is_removed'].sum()} items were removed.")
    print("-" * 50)

    base_name = os.path.splitext(os.path.basename(zip_filepath))[0] if zip_filepath else "reviewed_package"

    # --- Package 1: Ready for DC Review ---
    if not df_ready_for_review.empty:
        print("\n📦 Creating package for 'Ready for DC Review' items...")
        ready_zip_path = create_package(df_ready_for_review, "/content/ready_package", f"/content/{base_name}_READY_FOR_DC_REVIEW")
        print(f"✅ Successfully created '{os.path.basename(ready_zip_path)}'.")
        print("⬇️ Preparing download...")
        files.download(ready_zip_path)
    else:
        print("\nℹ️ No items were marked as 'Ready for DC Review'. Skipping package creation.")

    # --- Package 2: AI Processing To-Do ---
    if not df_ai_needed.empty and using_gdrive:
        print("\n🤖 Creating package for 'AI Processing' items...")
        ai_zip_base = os.path.join(GDRIVE_I2DC_PATH, f"{base_name}_AI_PROCESSING_TODO")
        ai_zip_path = create_package(df_ai_needed, "/content/ai_package", ai_zip_base, keep_ai_flags=True)
        print(f"✅ Successfully created '{os.path.basename(ai_zip_path)}'.")
        print(f"✅ Saved to your Google Drive in the 'i2dc' folder. You can now use this file with the AI processing notebooks.")
    elif not df_ai_needed.empty and not using_gdrive:
         print("\n🤖 To create the 'AI Processing' package, please use the Google Drive option in Step 1.")
    else:
        print("\nℹ️ No items were flagged for AI processing. Skipping package creation.")

    # Clean up session file
    if os.path.exists(SESSION_FILE_PATH):
        os.remove(SESSION_FILE_PATH)
        print("\n🧹 Cleaned up saved session file from Google Drive.")